In [1]:
import pandas as pd

In [2]:
vektis = pd.read_csv("../data/raw/Vektis Open Databestand Zorgverzekeringswet 2024 - gemeente.csv", sep=";")

In [3]:
vektis.head()

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15
1,M,0 t/m 4 jaar,AA EN HUNZE,503,462.82,594197.18,32444.90,35500.43,20536.52,15357.50,...,0.00,0.0,2453.22,0.00,0.00,0.0,23236.19,0.00,0.00,0.00
2,M,0 t/m 4 jaar,AALSMEER,731,665.75,1829500.58,55985.58,51112.38,25609.76,17045.38,...,0.00,0.0,24028.45,0.00,0.00,0.0,125837.16,0.00,0.00,0.00
3,M,0 t/m 4 jaar,AALTEN,662,596.80,862338.30,45558.63,45592.74,27044.28,7194.71,...,0.00,0.0,1549.40,0.00,0.00,0.0,123649.13,0.00,0.00,0.00
4,M,0 t/m 4 jaar,ACHTKARSPELEN,680,619.29,903357.49,41051.42,46985.28,21486.90,11042.53,...,0.00,0.0,1147.02,0.00,0.00,0.0,46237.53,0.00,0.00,0.00


## Step 1. Privacy bucket

In [4]:
# Privacy bucket: groups under 10 people (hidden so no one can be identified)
# plus people with unknown sex, age or municipality. All lumped into one unlabelled row.
bucket = vektis[vektis['gemeentenaam'].isna()]
bucket

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15


In [5]:
bucket_people = bucket["aantal_bsn"].sum()
total_people = vektis["aantal_bsn"].sum()
share = bucket_people / total_people * 100
print(f"Bucket: {bucket_people:,.0f} people, {share:.1f}% of all people")

Bucket: 412,394 people, 2.2% of all people


In [6]:
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [7]:
bucket_cost = bucket.filter(like="kosten").sum().sum()
cost_share = bucket_cost / total_costs * 100
print(f"Bucket: €{bucket_cost:,.0f}, {cost_share:.1f}% of all cost")

Bucket: €167,835,544, 0.3% of all cost


**Decision: privacy bucket**
- Found: one row with empty sex, age and municipality. Vektis puts groups under 10 people (privacy) and people with unknown sex, age or municipality here. It holds 412,394 people (2.2%) but only €167.8 million (0.3% of all cost).
- Did: removed the row. The raw file stays unchanged.
- Why: every KPI needs a municipality, and expected cost needs age and sex. This row has none of them, so it cannot be placed anywhere. Rejected: spreading it over municipalities, because we do not know where these people live.
- Effect: only 5 small cells are missing (Rozendaal, Schiermonnikoog, Vlieland). The bucket is cheap per person (about €760 per insured-year against €3,382), likely because many were insured for only part of the year. Removing it barely changes the national figures.


In [8]:
vektis.head()

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,NaN,NaN,NaN,412394,220748.91,75738481.61,11107600.71,6081467.62,2120101.39,1852201.82,...,233868.73,276735.8,1297283.78,53388.62,760262.82,0.0,489674.26,9488679.78,2134088.05,641623.15
1,M,0 t/m 4 jaar,AA EN HUNZE,503,462.82,594197.18,32444.90,35500.43,20536.52,15357.50,...,0.00,0.0,2453.22,0.00,0.00,0.0,23236.19,0.00,0.00,0.00
2,M,0 t/m 4 jaar,AALSMEER,731,665.75,1829500.58,55985.58,51112.38,25609.76,17045.38,...,0.00,0.0,24028.45,0.00,0.00,0.0,125837.16,0.00,0.00,0.00
3,M,0 t/m 4 jaar,AALTEN,662,596.80,862338.30,45558.63,45592.74,27044.28,7194.71,...,0.00,0.0,1549.40,0.00,0.00,0.0,123649.13,0.00,0.00,0.00
4,M,0 t/m 4 jaar,ACHTKARSPELEN,680,619.29,903357.49,41051.42,46985.28,21486.90,11042.53,...,0.00,0.0,1147.02,0.00,0.00,0.0,46237.53,0.00,0.00,0.00


In [9]:
vektis = vektis[vektis["gemeentenaam"].notna()]

In [10]:
vektis.shape

(12991, 31)

In [11]:
assert len(vektis) == 12991

In [12]:
# how many negative cells, and in which columns? 
neg_costs = costs < 0
neg_costs.sum().sum()

#  which rows are they, and how big?
neg_rows= vektis[(costs<0).any(axis=1)]

/var/folders/_m/ly5gvy212z7_t_8yklgv0z_w0000gn/T/ipykernel_90671/664716006.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  neg_rows= vektis[(costs<0).any(axis=1)]


In [13]:
vektis.head()

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,...,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
1,M,0 t/m 4 jaar,AA EN HUNZE,503,462.82,594197.18,32444.90,35500.43,20536.52,15357.50,...,0.0,0.0,2453.22,0.0,0.0,0.0,23236.19,0.0,0.0,0.0
2,M,0 t/m 4 jaar,AALSMEER,731,665.75,1829500.58,55985.58,51112.38,25609.76,17045.38,...,0.0,0.0,24028.45,0.0,0.0,0.0,125837.16,0.0,0.0,0.0
3,M,0 t/m 4 jaar,AALTEN,662,596.80,862338.30,45558.63,45592.74,27044.28,7194.71,...,0.0,0.0,1549.40,0.0,0.0,0.0,123649.13,0.0,0.0,0.0
4,M,0 t/m 4 jaar,ACHTKARSPELEN,680,619.29,903357.49,41051.42,46985.28,21486.90,11042.53,...,0.0,0.0,1147.02,0.0,0.0,0.0,46237.53,0.0,0.0,0.0
5,M,0 t/m 4 jaar,ALBLASSERDAM,566,511.47,1046813.52,59368.82,38480.42,23049.49,6932.46,...,0.0,0.0,109152.08,0.0,0.0,0.0,4065.44,0.0,0.0,0.0


In [14]:
row_total = costs.sum(axis=1)
town_total = row_total.groupby(vektis["gemeentenaam"]).sum()
town_total[["UITGEEST", "TWENTERAND"]]


gemeentenaam
UITGEEST      4.323860e+07
TWENTERAND    1.273882e+08
dtype: float64

**Decision: negative costs**
- Found: 2 negative cells out of about 338,000: Uitgeest, women 25-29, inpatient mental health stays (−€2,111), and Twenterand, women 30-34, birth care (−€2,300). Each is under 0.01% of that municipality's total cost.
- Did: kept them as they are.
- Why: they are real corrections (reversed or refunded claims), too small to move any result, and keeping them means totals match the official Vektis figures. Rejected: setting them to zero (totals would no longer match Vektis) and dropping the rows (would also remove 25 real cost cells per row).


In [15]:
vektis.columns


Index(['geslacht', 'leeftijdsklasse', 'gemeentenaam', 'aantal_bsn',
       'aantal_verzekerdejaren', 'kosten_medisch_specialistische_z',
       'kosten_farmacie', 'kosten_huisarts_inschrijftarief',
       'kosten_huisarts_consult', 'kosten_huisarts_mdz',
       'kosten_huisarts_overig', 'kosten_hulpmiddelen', 'kosten_mondzorg',
       'kosten_paramedische_zorg_fysioth', 'kosten_paramedische_zorg_overig',
       'kosten_ziekenvervoer_zittend', 'kosten_ziekenvervoer_liggend',
       'kosten_kraamzorg', 'kosten_verloskundige_zorg',
       'kosten_grensoverschrijdende_zorg', 'kosten_eerstelijns_ondersteuning',
       'kosten_geriatrische_revalidatiez', 'kosten_eerstelijnsverblijf',
       'kosten_verpleging_en_verzorging', 'kosten_gzsp',
       'kosten_integrale_geboortezorg', 'kosten_innovatiegelden_ggz',
       'kosten_overig', 'kosten_consulten_ggz',
       'kosten_intramuraal_verblijf_ggz', 'kosten_overige_prestaties_ggz'],
      dtype='str')

In [16]:
abroad = vektis["kosten_grensoverschrijdende_zorg"]
abroad_costs = abroad.sum()
abroad_share = abroad_costs / total_costs * 100
print(f"Care abroad: €{abroad_costs:,.0f}, {abroad_share:.1f}% of all cost")

Care abroad: €347,307,725, 0.6% of all cost


In [17]:
abroad_town = vektis["kosten_grensoverschrijdende_zorg"].groupby(vektis["gemeentenaam"]).sum()
abroad_pct = abroad_town / town_total * 100
abroad_pct.sort_values(ascending=False)
# abroad_pct.median()

gemeentenaam
SLUIS              18.394525
HULST              10.182345
TERNEUZEN           8.246225
VAALS               4.936495
VEERE               2.590779
                     ...    
STADSKANAAL         0.143058
AALTEN              0.127767
NEDER-BETUWE        0.115078
PEKELA              0.101732
SCHIERMONNIKOOG     0.085287
Length: 342, dtype: float64

**Decision: care abroad**
- Found: care abroad (`kosten_grensoverschrijdende_zorg`) is €347.3 million, 0.6% of all cost, and the leaflet says it is incomplete because foreign bills arrive late. It is concentrated in border municipalities: median 0.4% of a municipality's cost, but Sluis 18%, Hulst 10%, Terneuzen 8% (Zeeuws-Vlaanderen, next to Belgium) and Vaals 5% (next to Germany and Belgium).
- Did: kept the column.
- Why: for border towns, care abroad is part of their normal hospital care. Excluding it would make Sluis look about 18% cheaper than it is and give the purchasing team a false signal. Rejected: excluding the column.
- Limitation: because the column is incomplete, border towns' true cost is probably higher than shown. This goes in the README's Limitations.


In [25]:
vektis["leeftijdsklasse"].sort_values(ascending=False)

12991              90+
6383               90+
6391               90+
6390               90+
6389               90+
             ...      
6608     0 t/m  4 jaar
6607     0 t/m  4 jaar
6606     0 t/m  4 jaar
6605     0 t/m  4 jaar
6496     0 t/m  4 jaar
Name: leeftijdsklasse, Length: 12991, dtype: str

## Step 4. Age band as a number

In [24]:
sorted(vektis["leeftijdsklasse"].unique())

['0 t/m  4 jaar',
 '10 t/m 14 jaar',
 '15 t/m 19 jaar',
 '20 t/m 24 jaar',
 '25 t/m 29 jaar',
 '30 t/m 34 jaar',
 '35 t/m 39 jaar',
 '40 t/m 44 jaar',
 '45 t/m 49 jaar',
 '5 t/m  9 jaar',
 '50 t/m 54 jaar',
 '55 t/m 59 jaar',
 '60 t/m 64 jaar',
 '65 t/m 69 jaar',
 '70 t/m 74 jaar',
 '75 t/m 79 jaar',
 '80 t/m 84 jaar',
 '85 t/m 89 jaar',
 '90+']

In [ ]:
vektis["age_from"] = vektis["leeftijdsklasse"].str.extract(r"(\d+)").astype(int)

**Note: age order.** Age bands are text, so they sort wrongly ("5 t/m 9" after "45 t/m 49"). Added `age_from`, the lower age as a number (0, 5 ... 90), for ordering and charts. The text label stays for display.


In [ ]:
vektis[["leeftijdsklasse", "age_from"]].drop_duplicates().sort_values("age_from")


,leeftijdsklasse,age_from
1,0 t/m 4 jaar,0
343,5 t/m 9 jaar,5
685,10 t/m 14 jaar,10
1027,15 t/m 19 jaar,15
1369,20 t/m 24 jaar,20
1711,25 t/m 29 jaar,25
2053,30 t/m 34 jaar,30
2395,35 t/m 39 jaar,35
2737,40 t/m 44 jaar,40
3079,45 t/m 49 jaar,45
